# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from joblib import dump, load
from matplotlib import rc

# Data pre-processing

In [ ]:
# Create a loop that stores each data set into a dictionary
output = 'gisaxs_domain'
data_directory = '../../chemical-featurization/likeness_data/poly_maccs.xlsx'
df_likeness = pd.read_excel(data_directory)

# Bayesian-Search

In [ ]:
def parse_seed_indices(indices_csv="selected_indices.csv"):
    """Parse CSV containing index blocks separated by 'index for seed XXXX' headers."""
    seed_indices = {}  # {seed_id: [list of indices in acquisition order]}
    
    current_seed = None
    current_indices = []
    
    with open(indices_csv, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith("index for seed"):
                # Save previous seed block before starting new one
                if current_seed is not None:
                    seed_indices[current_seed] = current_indices
                current_seed = line.split("seed")[-1].strip()
                current_indices = []
            elif line.isdigit():
                current_indices.append(int(line))
    
    # Save the final seed block
    if current_seed is not None:
        seed_indices[current_seed] = current_indices
    
    return seed_indices


seed_indices = parse_seed_indices("selected_indices_350_10_seeds.csv")
print(f"Found {len(seed_indices)} seeds: {list(seed_indices.keys())}")
for seed, indices in seed_indices.items():
    print(f"  Seed {seed}: {len(indices)} indices")

In [ ]:
def bayes_train(df, output, n_samples=None, preloaded_indices=None):

    df_clean = df.dropna()
    inputs = df_clean.iloc[:, 1:-2].copy()
    target = df_clean[f"{output}"]
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        inputs, target, test_size=0.2, random_state=10
    )

    selected_indices = preloaded_indices
    
    if n_samples is not None:
        selected_indices = selected_indices[:n_samples]
    
    X_train = X_train_full.iloc[selected_indices]
    y_train = y_train_full.iloc[selected_indices]
    
    # Slice to first n_samples indices if specified
    if n_samples is not None:
        selected_indices = selected_indices[:n_samples]
    
    X_train = X_train_full.iloc[selected_indices]
    y_train = y_train_full.iloc[selected_indices]

    # Scale the inputs (fit only on training data to avoid data leakage)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
    X_test_scaled = scaler.transform(X_test)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

    RANDOM_SEED = 42
    CV_FOLDS = 5

    search_spaces = {

        # forest size (log scaling explores wide range efficiently)
        "n_estimators": Integer(50, 300),

        # tree complexity
        "max_depth": Integer(3, 20),

        # regularization knobs
        "min_samples_split": Integer(2, 20),
        "min_samples_leaf": Integer(1, 12),

        # feature sampling
        "max_features": Categorical(["sqrt", "log2", 1.0, 0.3, 0.5, 0.8]),

        # bootstrap behavior
        "bootstrap": Categorical([True, False]),

        # split criterion
        "criterion": Categorical(["squared_error", "friedman_mse"]),
    }

    rf = RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=60,                 # number of Bayesian steps
        scoring="r2",
        cv=CV_FOLDS,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbose=0
    )

    # Fit optimizer
    opt.fit(X_train_scaled, y_train)

    best_model = opt.best_estimator_
    best_cv_r2 = opt.best_score_
    best_test_r2 = best_model.score(X_test_scaled, y_test)

    print("CV Train R²:", best_cv_r2)
    print("Test R²:", best_test_r2)
    print("Best Parameters:", opt.best_params_)

    return best_model, scaler

In [ ]:
%%time

best_models = {}

for seed, indices in seed_indices.items():
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print(f"{'='*50}")
    # print(indices)
    for n in range(10, 351, 10):
        print(f"\n--- Seed {seed} | Training model with {n} samples ---")
        
        best_model, scaler = bayes_train(
            df_likeness, output,
            n_samples=n,
            preloaded_indices=indices
        )
        
        key = f"model_seed{seed}_n{n:03d}"
        best_models[key] = {"model": best_model, "scaler": scaler}
        
        dump(best_model, f"domain_indicies/rf_model_seed{seed}_n{n:03d}.pkl")
        dump(scaler,     f"domain_indicies/scaler_seed{seed}_n{n:03d}.pkl")

In [ ]:
validation_path = '../../chemical-featurization/likeness_data/validation/poly_maccs_validation_subset.xlsx'
# import data frame for pre-processing
df_val = pd.read_excel(validation_path)

In [ ]:
output = 'gisaxs_domain'

df_val_clean = df_val.dropna()
sample_names = df_val_clean["Sample"]
X_val = df_val_clean.iloc[:, 1:-2]
y_val = df_val_clean[f'{output}']

val_results = []

for seed in seed_indices.keys():
    print(f"\nSeed {seed}")
    for n in range(10, 351, 10):
        key = f"model_seed{seed}_n{n:03d}"
        model  = best_models[key]["model"]
        scaler = best_models[key]["scaler"]

        X_val_scaled = pd.DataFrame(
            scaler.transform(X_val), columns=X_val.columns, index=X_val.index
        )

        val_r2  = model.score(X_val_scaled, y_val)
        val_mae = mean_absolute_error(y_val, model.predict(X_val_scaled))

        val_results.append({"seed": seed, "n_samples": n, "val_r2": val_r2, "val_mae": val_mae})
        print(f"  n={n:3d} | Val R²: {val_r2:.4f} | Val MAE: {val_mae:.4f}")

val_results_df = pd.DataFrame(val_results)

r2_pivot = val_results_df.pivot(index="n_samples", columns="seed", values="val_r2")
r2_pivot.columns = [f"seed_{s}" for s in r2_pivot.columns]
r2_pivot["mean_r2"] = r2_pivot.mean(axis=1)
r2_pivot["std_r2"]  = r2_pivot.filter(like="seed_").std(axis=1)

print("\nValidation R² across seeds:")
print(r2_pivot.to_string())

In [ ]:
%matplotlib inline
rc('text', usetex=False)
rc('mathtext', fontset='cm')
rc('xtick', labelsize=25)   
rc('xtick.major', size=7)  
rc('xtick.minor', size=4)
rc('xtick.major', width=2)
rc('xtick.minor', width=2)
rc('ytick.major', width=2)
rc('ytick.minor', width=2)
rc('ytick', labelsize=25)
rc('ytick.major', size=7)
rc('ytick.minor', size=4)
rc('axes', labelsize=25) 
rc('axes', linewidth=2) 
rc('font',family='sans serif')
rc('font', style='normal')
rc('font', weight='500')
rc('font', size='15')
rc('axes', labelweight='500')
rc('axes.spines', **{'right':True, 'top':True}) 
plt.rcParams['font.family'] = 'Arial'

In [ ]:
# Aggregate the data as requested in the previous step
# Group by 'n', aggregate R2 for mean and std
r2_pivot = df.groupby('n')['R2'].agg(mean_r2='mean', std_r2='std')

# Sort by n just in case
r2_pivot = r2_pivot.sort_index()

# Now use the user's plotting code
fig, ax = plt.subplots(figsize=(8, 8))

# Plotting the mean line
ax.plot(r2_pivot.index, r2_pivot["mean_r2"], color="steelblue", linewidth=2.5, label=r"$\text{Mean } R^2$")

# Plotting the standard deviation shading
ax.fill_between(
    r2_pivot.index,
    r2_pivot["mean_r2"] - r2_pivot["std_r2"],
    r2_pivot["mean_r2"] + r2_pivot["std_r2"],
    alpha=0.15, color="steelblue", label=r"$\pm 1 \text{ SD}$"
)

# Formatting labels with LaTeX
ax.set_xlabel("Number of training samples")
ax.set_ylabel("Validation R$^2$")
ax.legend(loc="lower right", frameon=False)
ax.grid(False)
plt.tight_layout()

plt.savefig('learning_curve.pdf', format="pdf", transparent=True)